In [2]:
!pip install folium geopandas plotly scikit-learn


In [3]:
import pandas as pd
import numpy as np
import folium
import plotly.express as px
from sklearn.cluster import KMeans

In [4]:
data = {
    "Store": ["A","B","C","D","E"],
    "Latitude": [12.9716, 12.9352, 13.0358, 12.9900, 13.0500],
    "Longitude": [77.5946, 77.6245, 77.5970, 77.7000, 77.7200],
    "Sales": [50000, 30000, 80000, 20000, 15000]
}

df = pd.DataFrame(data)
df

,Store,Latitude,Longitude,Sales
0,A,12.9716,77.5946,50000
1,B,12.9352,77.6245,30000
2,C,13.0358,77.5970,80000
3,D,12.9900,77.7000,20000
4,E,13.0500,77.7200,15000


In [5]:
m = folium.Map(location=[12.97, 77.59], zoom_start=11)

for _, row in df.iterrows():
    folium.Marker(
        [row["Latitude"], row["Longitude"]],
        popup=f"{row['Store']} | Sales: {row['Sales']}"
    ).add_to(m)

m

In [6]:
df.sort_values("Sales", ascending=False)

,Store,Latitude,Longitude,Sales
2,C,13.0358,77.5970,80000
0,A,12.9716,77.5946,50000
1,B,12.9352,77.6245,30000
3,D,12.9900,77.7000,20000
4,E,13.0500,77.7200,15000


In [7]:
X = df[["Latitude", "Longitude"]]
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X)
df

,Store,Latitude,Longitude,Sales,Cluster
0,A,12.9716,77.5946,50000,0
1,B,12.9352,77.6245,30000,0
2,C,13.0358,77.5970,80000,0
3,D,12.9900,77.7000,20000,1
4,E,13.0500,77.7200,15000,1


In [8]:
fig = px.scatter_map(
    df,
    lat="Latitude",
    lon="Longitude",
    color="Cluster",
    size="Sales",
    hover_name="Store",
    zoom=10
)
fig.show()

In [10]:
centers = kmeans.cluster_centers_
centers_df = pd.DataFrame(
  centers, columns=["Latitude",
                    "Longitude"]
)
centers_df

,Latitude,Longitude
0,12.980867,77.605367
1,13.020000,77.710000


In [13]:
import pandas as pd

customers = pd.DataFrame({
    "Latitude":[12.97,12.98,12.99,13.00,13.01,13.02,13.03,13.04],
    "Longitude":[77.59,77.60,77.61,77.62,77.63,77.64,77.65,77.66],
    "Demand":[120,150,180,200,210,250,300,320]
})

customers


,Latitude,Longitude,Demand
0,12.97,77.59,120
1,12.98,77.60,150
2,12.99,77.61,180
3,13.00,77.62,200
4,13.01,77.63,210
5,13.02,77.64,250
6,13.03,77.65,300
7,13.04,77.66,320


In [14]:
from sklearn.cluster import KMeans
X = customers[["Latitude","Longitude"]]
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10

)
customers["Cluster"] = kmeans.fit_predict(X)
customers

,Latitude,Longitude,Demand,Cluster
0,12.97,77.59,120,2
1,12.98,77.60,150,2
2,12.99,77.61,180,0
3,13.00,77.62,200,0
4,13.01,77.63,210,0
5,13.02,77.64,250,1
6,13.03,77.65,300,1
7,13.04,77.66,320,1


In [19]:
cluster_demand = customers.groupby("Cluster")["Demand"].sum()

print(cluster_demand)

Cluster
0    590
1    870
2    270
Name: Demand, dtype: int64


In [20]:
store_locations = df[["Latitude","Longitude"]]

df["Cluster"] = kmeans.predict(store_locations)

store_count = df.groupby("Cluster").size()

print(store_count)

Cluster
0    1
1    2
2    2
dtype: int64


In [21]:
summary = pd.DataFrame({
    "Demand": cluster_demand,
    "Stores": store_count
}).fillna(0)

summary["Demand_per_Store"] = (
    summary["Demand"] / summary["Stores"]
)

summary.sort_values(
    "Demand_per_Store",
    ascending=False
)

,Demand,Stores,Demand_per_Store
Cluster,,,
0,590,1,590.0
1,870,2,435.0
2,270,2,135.0


In [23]:
best_cluster = summary["Demand_per_Store"].idxmax()

print(
    f"Recommended expansion cluster: {best_cluster}"
)

Recommended expansion cluster: 0
